# Where weather models miss: Seattle smoke and the 2026–27 El Niño

I don't have a background in climate modeling, so I started by asking where these models struggle. A friend in UW's Atmospheric Sciences PhD program pointed me to two kinds of cases: events driven by something the model doesn't represent, and conditions outside what the model was trained on. This notebook looks at one Seattle example of each, using NeuralGCM, Google's hybrid weather and climate model written in JAX.

1. **Wildfire smoke, September 2020.** The UW Climate Office [showed](https://climate.uw.edu/2020/10/07/cooler-temperatures-from-the-smoke-of-september/) that smoke made Washington days 10–20 °F colder than forecast. NeuralGCM has no smoke in it either. Does it miss in the same way?
2. **This winter's El Niño.** The Pacific is running about 3 °C warmer than normal, close to the strongest El Niños on record. What does NeuralGCM expect that to mean for a Seattle winter?

The setup is adapted from the official [NeuralGCM demo](https://colab.research.google.com/github/neuralgcm/neuralgcm/blob/main/docs/inference_demo.ipynb), and the framing draws on the [AIMIP Phase 1](https://arxiv.org/abs/2605.06944) comparison of AI climate models.

**To run:** Runtime → Change runtime type → T4 GPU. On a CPU, the El Niño section runs only a 1-day test.

In [ ]:
import importlib.util
if importlib.util.find_spec('neuralgcm') is None:
    %pip install -q -U neuralgcm dinosaur gcsfs

In [ ]:
import os, io, json, pickle, time, urllib.request
from concurrent.futures import ThreadPoolExecutor
import gcsfs, jax, jax.numpy as jnp, numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
from dinosaur import horizontal_interpolation, spherical_harmonic, xarray_utils
import neuralgcm

GPU = any(d.platform == 'gpu' for d in jax.devices())
FAST = not GPU                       # CPU: tiny run so the notebook still executes end to end
MODEL_NAME = 'v1_precip/stochastic_precip_2_8_deg.pkl' if GPU else 'v1/deterministic_2_8_deg.pkl'
MEMBERS = 3 if GPU else 1            # stochastic ensemble members per scenario
INIT = np.datetime64('2026-10-19T00')
END = INIT + np.timedelta64(1, 'D') if FAST else np.datetime64('2027-03-01T00')
FORCING_END = np.datetime64('2027-03-01T00')
GCS = dict(token='anon'); CACHE = 'cache'; os.makedirs(CACHE, exist_ok=True)
print(jax.devices(), '| FAST =', FAST, '|', MODEL_NAME)

## 0. A quick look at JAX
NeuralGCM is written in JAX. `jit` compiles a function the first time it runs and reuses the compiled version after that. `vmap` turns code written for one example into code that runs on a batch. For NeuralGCM, compiling is expensive: on my laptop the first one-hour model step took about 10 seconds and the next took about 3.

In [ ]:
f = jax.jit(lambda a: jnp.tanh(a @ a).sum())
x = jnp.ones((2048, 2048))
for label in ['compile + run', 'cached run']:
    t = time.time(); f(x).block_until_ready(); print(f'{label}: {time.time() - t:.3f}s')
batched = jax.vmap(lambda a: (a ** 2).mean())       # vmap: write per-example code, run batched
print(batched(jnp.arange(6.).reshape(3, 2)))

## 1. Load the model (from the demo)

In [ ]:
with gcsfs.GCSFileSystem(token='anon').open(f'gs://neuralgcm/models/{MODEL_NAME}', 'rb') as f:
    model = neuralgcm.PressureLevelModel.from_checkpoint(pickle.load(f))
full_era5 = xr.open_zarr('gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3',
                         chunks=None, storage_options=GCS)
print(model.input_variables, model.forcing_variables, model.timestep)

## 2. The smoke of September 2020

The blog compared what actually happened with GFS MOS, the standard forecast at the time. MOS corrects a weather model's output to match local weather stations, and it knew nothing about smoke. Observed minus forecast is therefore roughly the effect of the smoke.

I repeated that comparison and added a second forecast built from NeuralGCM:
- **GFS MOS**, downloaded from the Iowa Environmental Mesonet archive, to reproduce the blog.
- **NeuralGCM-MOS**: a simple regression that turns NeuralGCM's 24-hour forecast of air temperature a few thousand feet up into a station high and low. I fit it on July 15–August 31, before the smoke arrived. Google has already published its 2020 NeuralGCM forecasts, so this section does not need a GPU.
- **Observations** from the airport weather stations in Seattle (SEA), Omak (OMK), and Spokane (GEG).

In [ ]:
ST = {'SEA': (47.45, -122.31), 'OMK': (48.46, -119.52), 'GEG': (47.62, -117.53)}
obs = pd.read_csv('https://mesonet.agron.iastate.edu/cgi-bin/request/daily.py?network=WA_ASOS&'
                  + '&'.join(f'stations={s}' for s in ST)
                  + '&year1=2020&month1=7&day1=15&year2=2020&month2=9&day2=25&format=csv&var=max_temp_f&var=min_temp_f',
                  parse_dates=['day'])

# Map station to its exact GFS MOS station ID in IEM
MOS_STATION_MAP = {'SEA': 'KSEA', 'GEG': 'KGEG', 'OMK': 'KOMK'}

def gfs_mos_max(station, day):
    mos_id = MOS_STATION_MAP.get(station, f'K{station}')
    u = f'https://mesonet.agron.iastate.edu/api/1/mos.json?station={mos_id}&model=GFS&runtime={day:%Y-%m-%dT00:00Z}'
    try:
        req = urllib.request.urlopen(u, timeout=60)
        data = json.loads(req.read().decode('utf-8'))['data']
        v = [r['n_x'] for r in data if r['ftime'].startswith(f'{day + pd.Timedelta("1D"):%Y-%m-%d 00:00}')]
        return dict(station=station, day=day, gfs_mos=v[0] if v else np.nan)
    except Exception:
        return dict(station=station, day=day, gfs_mos=np.nan)

with ThreadPoolExecutor(8) as ex:
    mos = pd.DataFrame(ex.map(lambda a: gfs_mos_max(*a), [(s, d) for s in ST for d in pd.date_range('2020-09-01', '2020-09-25')]))

ens = xr.open_zarr('gs://weatherbench2/datasets/neuralgcm_ens/2020-240x121_equiangular_with_poles_conservative_mean.zarr',
                   chunks=None, storage_options=GCS)
days = pd.date_range('2020-07-15', '2020-09-25')
box = dict(latitude=slice(44, 52), longitude=slice(232, 246), level=[850, 925], prediction_timedelta=np.timedelta64(24, 'h'))
t_max = ens.temperature.sel(time=days.values, **box).compute()                          # valid 00Z next day
t_min = ens.temperature.sel(time=(days - pd.Timedelta('12h')).values, **box).compute()  # valid 12Z same day

In [ ]:
K2F = lambda k: (k - 273.15) * 9 / 5 + 32
rows = []
for s, (la, lo) in ST.items():
    px, pn = [K2F(da.interp(latitude=la, longitude=lo % 360)) for da in (t_max, t_min)]
    rows.append(pd.DataFrame({'station': s, 'day': days,
        'x925': px.sel(level=925).values, 'x850': px.sel(level=850).values,
        'n925': pn.sel(level=925).values, 'n850': pn.sel(level=850).values}))
df = pd.concat(rows).merge(obs, on=['station', 'day']).merge(mos, on=['station', 'day'], how='left')

def fit_predict(g, kind, target):   # tiny MOS: target ~ a + b*T925 + c*T850, trained pre-smoke
    X = lambda d: np.c_[np.ones(len(d)), d[f'{kind}925'], d[f'{kind}850']]
    cal = g[g.day <= '2020-08-31']
    coef = np.linalg.lstsq(X(cal), cal[target], rcond=None)[0]
    print(f"{g.station.iloc[0]} {target}: training RMSE {np.sqrt(((cal[target] - X(cal) @ coef) ** 2).mean()):.1f} °F")
    return X(g) @ coef
df = pd.concat([g.assign(ngcm_max=fit_predict(g, 'x', 'max_temp_f'), ngcm_min=fit_predict(g, 'n', 'min_temp_f'))
                for _, g in df.groupby('station')])
sep = df[df.day >= '2020-09-01'].assign(
    gap_gfs=lambda d: d.max_temp_f - d.gfs_mos, gap_ngcm=lambda d: d.max_temp_f - d.ngcm_max,
    gap_ngcm_min=lambda d: d.min_temp_f - d.ngcm_min)
sep.pivot(index='day', columns='station', values=['gap_gfs', 'gap_ngcm']).loc['2020-09-05':'2020-09-20'].round(1)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
for ax, (s, g) in zip(axes, sep.groupby('station')):
    ax.axvspan(pd.Timestamp('2020-09-11'), pd.Timestamp('2020-09-18'), color='0.85', label='heavy smoke')
    ax.plot(g.day, g.gap_gfs, 'o-', label='obs − GFS MOS (Bond 2020 baseline)')
    ax.plot(g.day, g.gap_ngcm, 's-', label='obs − NeuralGCM-MOS (Tmax)')
    ax.plot(g.day, g.gap_ngcm_min, '^:', alpha=.6, label='obs − NeuralGCM-MOS (Tmin)')
    ax.axhline(0, color='k', lw=.8); ax.set_ylabel('°F'); ax.set_title(s)
axes[0].legend(fontsize=8, ncol=2); plt.tight_layout()

**Conclusions**
- The GFS MOS comparison matches the blog. Forecasts were 12 °F too warm in Seattle on September 12, and 15 °F and 20 °F too warm in Omak and Spokane on the 13th.
- NeuralGCM missed the same days by more: 19–21 °F. Before the smoke, most of its errors were within 5 °F.
- Both forecasts miss for the same reason. Neither model knows the smoke is there, so the shared error is the smoke's effect. NeuralGCM's larger miss is partly my simple regression, which only had six weeks of clear summer days to learn from.

A model that is missing a process is not the same as a broken model. The error was large, but it was also consistent and easy to explain, and it points directly at the fix: give the model information about smoke.

## 3. The 2026–27 El Niño

Ocean temperatures drive much of a season's weather, so the experiment holds everything else fixed and changes only the ocean:
- Both runs start from the observed atmosphere on October 19, 2026 and run through February.
- **El Niño run:** normal ocean temperatures plus this October's observed warmth.
- **Neutral run:** the same, but with the tropical Pacific warmth removed.
- The difference between the two runs is NeuralGCM's estimate of what El Niño does to the winter. On a GPU, each run has three members with slightly different random noise, which shows how much of the difference is real.

Simplifications: the October warmth is held constant all winter, though real El Niños peak around December and then fade. The model grid is about 300 km wide, so "Seattle" here means the whole Pacific Northwest.

In [ ]:
def to_model_grid(ds, fill=True):
    src = spherical_harmonic.Grid(latitude_nodes=ds.sizes['latitude'], longitude_nodes=ds.sizes['longitude'],
                                  latitude_spacing=xarray_utils.infer_latitude_spacing(ds.latitude),
                                  longitude_offset=xarray_utils.infer_longitude_offset(ds.longitude))
    out = xarray_utils.regrid(ds, horizontal_interpolation.ConservativeRegridder(src, model.data_coords.horizontal, skipna=True))
    return xarray_utils.fill_nan_with_nearest(out) if fill else out

def initial_condition():
    path = f'{CACHE}/ic_{str(INIT)[:10]}.nc'
    if not os.path.exists(path):
        to_model_grid(full_era5[model.input_variables + model.forcing_variables].sel(time=[INIT]).compute()).to_netcdf(path)
    return xr.open_dataset(path)

def month_pos(d):  # fractional month index; 0 == mid-January
    d = d.astype('datetime64[D]'); m0 = d.astype('datetime64[M]')
    frac = (d - m0).astype(int) / ((m0 + 1).astype('datetime64[D]') - m0.astype('datetime64[D]')).astype(int)
    return (m0.astype(int) % 12) + frac - 0.5

def build_forcings():
    path = f'{CACHE}/forcings_2026-27.nc'
    if os.path.exists(path): return xr.open_dataset(path)
    wb2 = xr.open_zarr('gs://weatherbench2/datasets/era5/1959-2022-6h-128x64_equiangular_with_poles_conservative.zarr',
                       chunks=None, storage_options=GCS)
    clim_src = (wb2[['sea_surface_temperature', 'sea_ice_cover']].sel(time=slice('1991', '2020')).compute()
                .groupby('time.month').mean().rename(month='time').assign_coords(time=np.arange(12))
                .transpose('time', 'longitude', 'latitude'))
    wrap = lambda c: xr.concat([c, c.isel(time=0).assign_coords(time=12)], 'time')
    clim, clim_ocean = wrap(to_model_grid(clim_src)), wrap(to_model_grid(clim_src, fill=False))  # ocean-only for anomalies
    days = np.arange(INIT, FORCING_END + np.timedelta64(1, 'D'), np.timedelta64(1, 'D')).astype('datetime64[D]')
    pos = np.array([month_pos(d) for d in days]) % 12
    daily = (clim.interp(time=xr.DataArray(pos, dims='t')).drop_vars('time').rename(t='time')
             .assign_coords(time=days.astype('datetime64[ns]')))
    obs_t = np.arange(np.datetime64('2026-10-01T00'), INIT + np.timedelta64(1, 'h'), np.timedelta64(24, 'h'))
    obs = to_model_grid(full_era5[['sea_surface_temperature']].sel(time=obs_t).compute()
                        .transpose('time', 'longitude', 'latitude'), fill=False).mean('time')
    anom = obs.sea_surface_temperature - clim_ocean.sea_surface_temperature.interp(time=np.mean([month_pos(d) for d in obs_t]) % 12)
    taper = lambda x, full, zero: xr.where(abs(x) <= full, 1.0, xr.where(abs(x) >= zero, 0.0,
                                           0.5 * (1 + np.cos(np.pi * (abs(x) - full) / (zero - full)))))
    anom = (anom * taper(anom.latitude, 50, 60)).fillna(0.0)          # leave ice-edge SST at climatology
    tp_mask = taper(anom.latitude, 15, 25) * taper(anom.longitude - 202.5, 72.5, 82.5)
    ds = xr.Dataset({'sst_elnino': daily.sea_surface_temperature + anom,
                     'sst_neutral': daily.sea_surface_temperature + anom * (1 - tp_mask),
                     'sea_ice_cover': daily.sea_ice_cover, 'sst_anomaly_oct2026': anom, 'tropical_pacific_mask': tp_mask})
    ds.to_netcdf(path); return ds

ic, forc = initial_condition(), build_forcings()
a = forc.sst_anomaly_oct2026
print(f"Niño3.4 anomaly, Oct 1–19 2026: {float(a.sel(latitude=slice(-5, 5)).where((a.longitude >= 190) & (a.longitude <= 240)).mean()):+.2f} °C")
(forc.sst_elnino - forc.sst_neutral).isel(time=0).plot(x='longitude', y='latitude', cmap='RdBu_r', center=0, size=3, aspect=2.5)
plt.title('SST difference, El Niño − neutral run (K)');

In [ ]:
G = 9.80665
def reduce_output(ds):   # keep a few 2-D fields so a whole winter fits in memory
    at = lambda da, l: da.sel(level=l, drop=True)
    up = ds.sel(level=ds.level[ds.level >= 300])
    vint = lambda x: x.integrate('level') * 100.0 / G     # hPa -> Pa
    return xr.Dataset({'t925': at(ds.temperature, 925), 't850': at(ds.temperature, 850),
                       'z500': at(ds.geopotential, 500) / G, 'u200': at(ds.u_component_of_wind, 200),
                       'tcwv': vint(up.specific_humidity),
                       'ivt': np.hypot(vint(up.specific_humidity * up.u_component_of_wind),
                                       vint(up.specific_humidity * up.v_component_of_wind))})

dt, chunk = np.timedelta64(12, 'h'), 14                    # save every 12 h, 7-day chunks
inputs, in_forc = model.inputs_from_xarray(ic.isel(time=0)), model.forcings_from_xarray(ic.isel(time=0))
results = {}
for scen in ['elnino', 'neutral']:
    all_forc = model.forcings_from_xarray(xr.Dataset({'sea_surface_temperature': forc[f'sst_{scen}'],
                                                      'sea_ice_cover': forc.sea_ice_cover}).transpose('time', 'longitude', 'latitude'))
    members = []
    for m in range(MEMBERS):
        state, t, parts, tic = model.encode(inputs, in_forc, jax.random.key(m)), INIT, [], time.time()
        while t < END:
            steps = min(chunk, int((END - t) / dt))
            state, pred = model.unroll(state, all_forc, steps=steps, timedelta=dt, start_with_input=False)
            times = t + dt * np.arange(1, steps + 1)
            parts.append(reduce_output(model.data_to_xarray(pred, times=times))); t = times[-1]
        members.append(xr.concat(parts, 'time')); print(scen, m, f'{time.time() - tic:.0f}s')
    results[scen] = xr.concat(members, 'member')
    results[scen].to_netcdf(f'{scen}_{str(INIT)[:10]}.nc')   # download these before the Colab session ends

In [ ]:
win = slice(None) if FAST else slice('2026-12-01', '2027-02-28')
diff = (results['elnino'] - results['neutral']).sel(time=win).mean('time')    # per member
signal, spread = diff.mean('member'), diff.std('member') if MEMBERS > 1 else None
na = dict(longitude=slice(150, 300), latitude=slice(10, 75))
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (v, lab) in zip(axes, [('t850', 'T850 (K)'), ('z500', 'Z500 (m)'), ('ivt', 'IVT (kg/m/s)')]):
    signal[v].sel(**na).plot(x='longitude', y='latitude', ax=ax, cmap='RdBu_r', center=0, robust=True)
    ax.plot(360 - 122.3, 47.6, 'k*', ms=10); ax.set_title(f'El Niño − neutral: {lab}')
plt.tight_layout()

boxes = {'Pacific NW (42–49N, 125–115W)': dict(latitude=slice(42, 49), longitude=slice(235, 245)),
         'Aleutian Low (40–60N, 180–140W)': dict(latitude=slice(40, 60), longitude=slice(180, 220)),
         'US Southeast (28–36N, 95–80W)': dict(latitude=slice(28, 36), longitude=slice(265, 280))}
pd.DataFrame({name: {v: f"{float(signal[v].sel(**b).mean()):+.2f}" + (f" ± {float(spread[v].sel(**b).mean()):.2f}" if spread is not None else '')
                     for v in ['t925', 't850', 'z500', 'ivt']} for name, b in boxes.items()}).T

**Conclusions**

A typical strong El Niño winter brings a deeper storm system over the North Pacific, a warmer and drier Pacific Northwest, and a wetter US Southeast. NOAA's outlook for this winter expects the same. The open question is whether NeuralGCM reproduces this pattern for an El Niño this strong, since only two comparable events (1982–83 and 1997–98) fall inside its training data.

*Results to be added after the full GPU run.* ERA5 data for December–February will be available in spring 2027, and comparing it against these runs will be a true out-of-sample test.

## 4. Next questions

1. **Add smoke to the model.** If NeuralGCM is given smoke data as an input, does the September 2020 error go away without hurting forecasts on clear days?
2. **Test on other smoke years.** Does a correction learned from 2020 also work in 2017, 2018, 2021, and 2023?
3. **Let El Niño change over the winter.** How much does the Seattle result depend on holding October's ocean temperatures fixed?
4. **Push past the training data.** If the El Niño warmth is scaled from half to double, where does the model's response stop making sense?
5. **Check the uncertainty.** When the model is wrong, does the spread between runs grow to reflect that?

## 5. If models fail during unusual events, how can we trust them?

These models are most useful exactly when conditions are unusual, so this is the right question to ask. My view after this project is that trust depends on why a model fails.

**Missing information vs. a broken model.** In 2020, every forecast model missed the smoke because none of them had smoke data. The error was large, but it was also predictable and easy to explain. A model whose failures can be traced to a specific cause can be trusted within known limits. The more serious failure is a forecast that looks reasonable, is presented with confidence, and is wrong.

**Weather, seasons, and climate are different questions.** Day-to-day weather can only be forecast about two weeks ahead, no matter how good the model is, because small errors grow until they swamp the signal. A seasonal outlook asks a different question: not what happens on January 12, but whether the winter is more likely to be warm or wet. Climate projections ask how that whole range shifts as the planet warms. No model predicts the unpredictable. A good one tells you how uncertain it is.

**What AIMIP found.** AIMIP trained AI climate models on 1979–2014 and tested them on 2015–2024. Most underestimated recent warming. When the oceans were made 4 °C warmer, their answers diverged widely, and some even predicted cooling over land. The models built on physics, including NeuralGCM, held up best. The authors call generalizing to new conditions "a work in progress."

For me, that leads to four rules:
- Trust a model in conditions like the ones it was trained on, and ask for evidence outside them.
- Prefer models with physics built in, since physical laws still hold when conditions change.
- Rely on a range of runs rather than a single one.
- Keep traditional physics models as a check.

As the climate changes, unusual events become more common, and these questions matter more.